# 📊 Enrichissement avec Yahoo Finance (yfinance)

Ce notebook enrichit les Key Market Data avec des données en temps réel depuis Yahoo Finance.

**Objectif :**
- Enrichir toutes les entreprises (ou un sous-ensemble) avec yfinance
- Ajouter prix actuel, Beta (volatilité), Market Cap, etc.
- Ajouter Weight S&P 500 depuis CSV si disponible
- Sauvegarder les données enrichies dans `all_market_data.json`

**Pourquoi un script séparé (`enrich_with_yfinance.py`) ?**
- ✅ **Réutilisable** : Peut être importé dans `generate_company_universe.ipynb`
- ✅ **Modulaire** : Fonctions testées et réutilisables
- ✅ **Rapide** : Import direct sans re-exécuter tout le notebook


## 1. Imports et Configuration


In [ ]:
import json
from pathlib import Path
import sys
import pandas as pd

# Ajouter scripts au path
BASE_DIR = Path('../../')
sys.path.append(str(BASE_DIR / 'scripts'))

# Importer fonction d'enrichissement
from enrich_with_yfinance import enrich_ticker_with_yfinance, enrich_multiple_tickers

# Chemin Key Market Data
KEY_MARKET_DATA_FILE = BASE_DIR / 'data' / 'generated' / 'key_market_data' / 'all_market_data.json'

print(f"📁 Fichier Key Market Data: {KEY_MARKET_DATA_FILE}")


## 2. Charger Key Market Data


In [ ]:
print("📊 Chargement Key Market Data...")

with open(KEY_MARKET_DATA_FILE, 'r', encoding='utf-8') as f:
    key_market_data = json.load(f)

print(f"✅ {len(key_market_data)} entreprises chargées")

# Afficher quelques exemples
print("\n📋 Exemples de tickers disponibles:")
example_tickers = list(key_market_data.keys())[:10]
for ticker in example_tickers:
    price = key_market_data[ticker].get('current_price', 'N/A')
    print(f"   {ticker}: ${price}" if price != 'N/A' else f"   {ticker}: N/A")


## 3. Test Simple - Un Seul Ticker


In [ ]:
# Test avec Apple (AAPL)
test_ticker = "AAPL"

if test_ticker in key_market_data:
    csv_data = key_market_data[test_ticker]
    csv_price = csv_data.get('current_price', 0)
    
    print(f"📊 Test avec {test_ticker}")
    print(f"   Prix CSV (15 août): ${csv_price}")
    print(f"\n🔄 Enrichissement avec Yahoo Finance...")
    
    # Enrichir
    enrichment = enrich_ticker_with_yfinance(test_ticker, csv_price)
    
    if enrichment.get('success'):
        print(f"\n✅ Enrichissement réussi!")
        print(f"\n📊 Résultats:")
        print(f"   Prix CSV: ${enrichment['csv_price']:.2f}")
        print(f"   Prix actuel: ${enrichment['current_price']:.2f}")
        print(f"   Variation: {enrichment['price_change_pct']:+.2f}%")
        print(f"   Beta (volatilité): {enrichment['beta']:.2f}")
        print(f"   Facteur volatilité: {enrichment['volatility_factor']:.3f}")
        print(f"   Risque volatilité: {enrichment['volatility_risk']}")
        print(f"   Market Cap: ${enrichment.get('market_cap', 0):,.0f}")
        print(f"\n📋 Données complètes:")
        print(json.dumps(enrichment, indent=2, ensure_ascii=False))
    else:
        print(f"\n❌ Erreur: {enrichment.get('error', 'Unknown')}")
        print(json.dumps(enrichment, indent=2))
else:
    print(f"⚠️ {test_ticker} non trouvé dans Key Market Data")


## 4. Enrichissement Complet - Toutes les Entreprises


In [ ]:
# Configuration de l'enrichissement
ENRICH_ALL = False  # Mettre True pour enrichir toutes les entreprises (peut être long!)
ENRICH_TOP_N = 50  # Enrichir seulement les N premières (None = désactiver)
ENRICH_SP500_ONLY = False  # Enrichir seulement les tickers S&P 500

# Préparer la liste des tickers à enrichir
tickers_to_enrich = []

if ENRICH_ALL:
    # Enrichir toutes les entreprises
    tickers_to_enrich = list(key_market_data.keys())
    print(f"📊 Mode: Enrichissement de TOUTES les entreprises ({len(tickers_to_enrich)})")
elif ENRICH_TOP_N:
    # Enrichir seulement les N premières
    tickers_to_enrich = list(key_market_data.keys())[:ENRICH_TOP_N]
    print(f"📊 Mode: Enrichissement des {ENRICH_TOP_N} premières entreprises")
elif ENRICH_SP500_ONLY:
    # Enrichir seulement les S&P 500
    tickers_to_enrich = [t for t, data in key_market_data.items() 
                        if data.get('is_sp500', False)]
    print(f"📊 Mode: Enrichissement seulement S&P 500 ({len(tickers_to_enrich)} entreprises)")
else:
    print("⚠️ Enrichissement désactivé. Modifier ENRICH_TOP_N ou ENRICH_ALL pour activer.")
    
if tickers_to_enrich:
    print(f"📋 {len(tickers_to_enrich)} tickers à enrichir")
    print(f"⏱️  Temps estimé: ~{len(tickers_to_enrich) * 0.5 / 60:.1f} minutes (0.5s par ticker)")


In [ ]:
# Enrichir toutes les entreprises sélectionnées
if tickers_to_enrich:
    print("\n🔄 Début de l'enrichissement...")
    print("="*80)
    
    # Préparer les prix CSV (utiliser current_price si disponible, sinon 0)
    csv_prices = {}
    for ticker in tickers_to_enrich:
        csv_price = key_market_data[ticker].get('current_price', 0)
        if csv_price is None:
            csv_price = 0
        csv_prices[ticker] = csv_price
    
    # Enrichir avec délai pour éviter rate limiting
    enriched_results = enrich_multiple_tickers(
        tickers_to_enrich, 
        csv_prices, 
        delay=0.5,  # 0.5 seconde entre chaque requête
        max_tickers=None
    )
    
    print(f"\n✅ Enrichissement terminé!")
    print(f"   - Succès: {sum(1 for r in enriched_results.values() if r.get('success'))}")
    print(f"   - Échecs: {sum(1 for r in enriched_results.values() if not r.get('success'))}")
    
    # Intégrer les résultats enrichis dans key_market_data
    for ticker, enrichment in enriched_results.items():
        if ticker in key_market_data:
            # Créer ou mettre à jour la section realtime
            if 'market_data' not in key_market_data[ticker]:
                key_market_data[ticker]['market_data'] = {}
            key_market_data[ticker]['market_data']['realtime'] = enrichment
            
            # Mettre à jour current_price si enrichi avec succès
            if enrichment.get('success') and enrichment.get('current_price'):
                key_market_data[ticker]['current_price'] = enrichment['current_price']
            
            # Ajouter Weight S&P 500 si disponible
            if enrichment.get('sp500_weight') is not None:
                key_market_data[ticker]['sp500_weight'] = enrichment['sp500_weight']
                key_market_data[ticker]['is_sp500'] = enrichment.get('is_sp500', False)
else:
    print("⚠️ Aucun ticker à enrichir")


## 5. Sauvegarder les Données Enrichies


In [ ]:
# Sauvegarder les Key Market Data enrichis
if tickers_to_enrich:
    output_file = BASE_DIR / 'data' / 'generated' / 'key_market_data' / 'all_market_data.json'
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(key_market_data, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Fichier sauvegardé: {output_file}")
    print(f"📊 {len(key_market_data)} entreprises dans le fichier")
    
    # Statistiques finales
    enriched_count = sum(1 for ticker in key_market_data.values() 
                        if ticker.get('market_data', {}).get('realtime', {}).get('success', False))
    print(f"📈 {enriched_count} entreprises enrichies avec succès")
    
    # Sauvegarder aussi une version avec timestamp
    from datetime import datetime
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_file = BASE_DIR / 'data' / 'generated' / 'key_market_data' / f'all_market_data_enriched_{timestamp}.json'
    with open(backup_file, 'w', encoding='utf-8') as f:
        json.dump(key_market_data, f, indent=2, ensure_ascii=False)
    print(f"💾 Backup sauvegardé: {backup_file}")


## 6. Exemples de Données Enrichies


In [ ]:
# Afficher quelques exemples d'enrichissement
if tickers_to_enrich:
    print("📊 Exemples d'enrichissement:\n")
    
    example_count = 0
    for ticker, data in key_market_data.items():
        realtime = data.get('market_data', {}).get('realtime', {})
        if realtime.get('success') and example_count < 5:
            print(f"\n{'='*80}")
            print(f"📈 {ticker} - {data.get('company_name', 'N/A')}")
            print(f"{'='*80}")
            
            if realtime.get('current_price'):
                csv_price = realtime.get('csv_price', 0)
                current_price = realtime.get('current_price', 0)
                change = realtime.get('price_change_pct', 0)
                print(f"💰 Prix: ${csv_price:.2f} → ${current_price:.2f} ({change:+.2f}%)")
            
            if realtime.get('beta'):
                print(f"📊 Beta (volatilité): {realtime.get('beta'):.2f}")
                print(f"   Risque: {realtime.get('volatility_risk', 'N/A')}")
                print(f"   Facteur: {realtime.get('volatility_factor', 1.0):.3f}")
            
            if realtime.get('sp500_weight'):
                print(f"⚖️  Weight S&P 500: {realtime.get('sp500_weight')*100:.4f}%")
            
            example_count += 1
    
    if example_count == 0:
        print("⚠️ Aucune entreprise enrichie avec succès")


## 4. Test avec Plusieurs Tickers (Top 10)


In [ ]:
# Sélectionner 10 tickers avec prix valides
valid_tickers = []
for ticker, data in key_market_data.items():
    price = data.get('current_price')
    if price and price > 0:
        valid_tickers.append(ticker)
    if len(valid_tickers) >= 10:
        break

print(f"📊 Test avec {len(valid_tickers)} tickers: {valid_tickers}")

# Extraire prix CSV
csv_prices = {}
for ticker in valid_tickers:
    csv_prices[ticker] = key_market_data[ticker].get('current_price', 0)

print(f"\n🔄 Enrichissement en cours (délai 0.5s entre requêtes)...")
print(f"⏱️  Temps estimé: ~{len(valid_tickers) * 0.5:.1f} secondes\n")

# Enrichir
results = enrich_multiple_tickers(valid_tickers, csv_prices, delay=0.5)

print(f"\n✅ Enrichissement terminé!")


## 5. Comparaison CSV vs Prix Actuel


In [ ]:
# Créer DataFrame de comparaison
comparison_data = []

for ticker, enrichment in results.items():
    if enrichment.get('success'):
        comparison_data.append({
            'ticker': ticker,
            'csv_price': enrichment['csv_price'],
            'current_price': enrichment['current_price'],
            'price_change_pct': enrichment['price_change_pct'],
            'beta': enrichment['beta'],
            'volatility_factor': enrichment['volatility_factor'],
            'volatility_risk': enrichment['volatility_risk']
        })

if comparison_data:
    df = pd.DataFrame(comparison_data)
    df = df.sort_values('price_change_pct', ascending=False)
    
    print("📊 Comparaison Prix CSV vs Prix Actuel")
    print("=" * 80)
    print(df.to_string(index=False))
    
    # Statistiques
    print(f"\n📈 Statistiques:")
    print(f"   Variation moyenne: {df['price_change_pct'].mean():+.2f}%")
    print(f"   Variation min: {df['price_change_pct'].min():+.2f}%")
    print(f"   Variation max: {df['price_change_pct'].max():+.2f}%")
    print(f"   Beta moyen: {df['beta'].mean():.2f}")
    print(f"   Entreprises très volatiles (Beta > 1.5): {len(df[df['beta'] > 1.5])}")
else:
    print("⚠️ Aucune donnée enrichie disponible")


## 6. Test Utilisation dans Analyse d'Impact


In [ ]:
# Simuler calcul d'impact avec enrichissement
def simulate_impact_analysis(ticker: str, base_risk_score: float, impact_pct: float):
    """
    Simule calcul d'impact réglementaire avec et sans enrichissement
    """
    if ticker not in key_market_data:
        return None
    
    csv_price = key_market_data[ticker].get('current_price', 0)
    
    # Sans enrichissement (CSV seulement)
    impact_csv = csv_price * (impact_pct / 100)
    
    # Avec enrichissement
    enrichment = enrich_ticker_with_yfinance(ticker, csv_price)
    
    if enrichment.get('success'):
        current_price = enrichment['current_price']
        impact_current = current_price * (impact_pct / 100)
        adjusted_risk = base_risk_score * enrichment['volatility_factor']
        
        return {
            'ticker': ticker,
            'base_risk_score': base_risk_score,
            'adjusted_risk_score': round(adjusted_risk, 2),
            'impact_pct': impact_pct,
            'csv_price': csv_price,
            'current_price': current_price,
            'impact_csv': round(impact_csv, 2),
            'impact_current': round(impact_current, 2),
            'difference': round(impact_current - impact_csv, 2),
            'beta': enrichment['beta'],
            'volatility_factor': enrichment['volatility_factor']
        }
    
    return None

# Test avec quelques entreprises
test_cases = [
    ("AAPL", 65, -5.0),   # Apple, risque 65, impact -5%
    ("MSFT", 70, -3.0),   # Microsoft, risque 70, impact -3%
    ("GOOGL", 60, -4.0)   # Google, risque 60, impact -4%
]

print("🧪 Test Analyse d'Impact avec Enrichissement\n")
print("=" * 80)

for ticker, base_risk, impact_pct in test_cases:
    result = simulate_impact_analysis(ticker, base_risk, impact_pct)
    
    if result:
        print(f"\n📊 {result['ticker']}:")
        print(f"   Risque base: {result['base_risk_score']}/100")
        print(f"   Risque ajusté (Beta): {result['adjusted_risk_score']}/100")
        print(f"   Beta: {result['beta']:.2f} → Facteur: {result['volatility_factor']:.3f}")
        print(f"\n   Impact (-{abs(result['impact_pct'])}%):")
        print(f"      Avec CSV (${result['csv_price']:.2f}): -${abs(result['impact_csv']):.2f}")
        print(f"      Avec prix actuel (${result['current_price']:.2f}): -${abs(result['impact_current']):.2f}")
        print(f"      Différence: ${abs(result['difference']):.2f} (plus précis avec yfinance)")
        print("\n" + "-" * 80)
    else:
        print(f"⚠️ {ticker}: Erreur ou non disponible")


## 7. Conclusion

**Ce que fait yfinance :**
- ✅ Ajoute prix actuel (vs CSV du 15 août)
- ✅ Ajoute Beta (volatilité) pour ajuster risque
- ✅ Calcule facteur de volatilité
- ✅ Ne remplace PAS les données CSV (base de référence)

**Utilité :**
- 💰 Impact financier plus précis (avec prix actuel)
- 📊 Risque ajusté selon volatilité (Beta)
- 📝 Justifications plus transparentes

**Prochaines étapes :**
- Intégrer dans `generate_company_universe.ipynb`
- Utiliser dans `impact_analysis.py`
- Afficher dans Streamlit dashboard
